# 🎧 NexTune — EDA Notebook
**219 products | 40 features | Amazon India Bluetooth Headphones**

> Colab: Runtime → Run all

In [ ]:
!pip install -q pandas numpy matplotlib seaborn
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings, os
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 5)

URL = 'https://raw.githubusercontent.com/ESpoorthy/NexTune/main/data/final-merged-dataset.csv' \
      if os.path.exists('/content') else '../data/final-merged-dataset.csv'
df = pd.read_csv(URL)

# Fix numeric types
num_cols = ['price_inr','rating','review_count','battery_life_hrs','bluetooth_version',
            'driver_size_mm','mic_count','anc_db','ipx_level','charging_time_hrs',
            'latency_ms','range_m','eq_modes']
bin_cols = ['has_noise_cancellation','has_enc','has_usb_c','has_premium_codec',
            'has_touch_control','has_voice_assistant','has_fast_charge','has_dual_pairing',
            'has_gaming_mode','has_hi_res_audio','has_spatial_audio','has_low_latency']
for c in num_cols + bin_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
for c in bin_cols:
    if c in df.columns:
        df[c] = df[c].fillna(0).astype(int)

print(f"✅ Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(3)

## 1. Dataset Overview

In [ ]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"  Products          : {len(df)}")
print(f"  Features          : {len(df.columns)}")
print(f"  Unique brands     : {df['brand'].nunique()}")
print(f"  Price range       : ₹{df['price_inr'].min():.0f} – ₹{df['price_inr'].max():.0f}")
print(f"  Median price      : ₹{df['price_inr'].median():.0f}")
print(f"  Mean price        : ₹{df['price_inr'].mean():.0f}")
print(f"  Avg rating        : {df['rating'].mean():.2f} / 5.0  ({df['rating'].notna().sum()} rated)")
print(f"  With country data : {df['country_of_origin'].notna().sum()}")
print(f"  With parent co.   : {df['parent_company'].notna().sum()}")
print("=" * 60)
print("\nMissing values per column:")
miss = df.isnull().sum()
print(miss[miss > 0].sort_values(ascending=False).to_string())

## 2. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df['price_inr'].dropna(), bins=40, color='#378ADD', edgecolor='white')
axes[0].axvline(df['price_inr'].median(), color='red',    ls='--', lw=2, label=f"Median ₹{df['price_inr'].median():.0f}")
axes[0].axvline(df['price_inr'].mean(),   color='orange', ls='--', lw=2, label=f"Mean ₹{df['price_inr'].mean():.0f}")
axes[0].set_title('Price Distribution', fontweight='bold'); axes[0].set_xlabel('Price (INR)'); axes[0].legend()

axes[1].boxplot(df['price_inr'].dropna(), patch_artist=True, boxprops=dict(facecolor='#378ADD', alpha=0.6),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Price Box Plot', fontweight='bold'); axes[1].set_ylabel('Price (INR)')

axes[2].hist(np.log1p(df['price_inr'].dropna()), bins=40, color='#1D9E75', edgecolor='white')
axes[2].set_title('Price — Log Scale', fontweight='bold'); axes[2].set_xlabel('log(1 + Price)')

plt.suptitle('Price Analysis  |  219 Products', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print(df['price_inr'].describe().round(0).to_string())

## 3. Category & Brand Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

cat = df['category'].value_counts()
axes[0].bar(cat.index, cat.values, color=['#378ADD','#1D9E75','#D85A30','#BA7517'])
axes[0].set_title('Products by Category', fontweight='bold'); axes[0].tick_params(axis='x', rotation=15)
for i,v in enumerate(cat.values): axes[0].text(i, v+1, str(v), ha='center', fontweight='bold')

df.boxplot(column='price_inr', by='category', ax=axes[1],
           boxprops=dict(color='#378ADD'), medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Price by Category', fontweight='bold'); axes[1].set_xlabel(''); plt.sca(axes[1]); plt.xticks(rotation=15); plt.suptitle('')

top = df['brand'].value_counts().head(12)
axes[2].barh(top.index[::-1], top.values[::-1], color='#378ADD')
axes[2].set_title('Top 12 Brands', fontweight='bold'); axes[2].set_xlabel('Products')

plt.tight_layout(); plt.show()
print("\nCategory price stats:")
print(df.groupby('category')['price_inr'].agg(['count','mean','median','min','max']).round(0).to_string())

## 4. Country of Origin & Parent Company

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

country = df['country_of_origin'].fillna('Unknown').value_counts()
axes[0].pie(country.values, labels=country.index, autopct='%1.1f%%',
            colors=['#378ADD','#D85A30','#1D9E75','#BA7517','#D4537E','#888888'], startangle=90)
axes[0].set_title('Products by Country of Origin', fontweight='bold')

cp = df.groupby('country_of_origin')['price_inr'].median().sort_values(ascending=False)
axes[1].bar(cp.index, cp.values, color='#1D9E75')
axes[1].set_title('Median Price by Country', fontweight='bold'); axes[1].set_ylabel('Median Price (INR)')
axes[1].tick_params(axis='x', rotation=15)
for i,v in enumerate(cp.values): axes[1].text(i, v+30, f'₹{v:.0f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout(); plt.show()
print("\nCountry breakdown:")
print(df.groupby('country_of_origin')['price_inr'].agg(['count','mean','median']).round(0).to_string())
print("\nTop parent companies:")
print(df['parent_company'].value_counts().head(10).to_string())

## 5. All 18 Scraped Features — Coverage

In [ ]:
features_18 = {
    # Numerical (10)
    'battery_life_hrs':   'Battery Life (hrs)',
    'bluetooth_version':  'Bluetooth Version',
    'driver_size_mm':     'Driver Size (mm)',
    'mic_count':          'Mic Count',
    'anc_db':             'ANC Level (dB)',
    'ipx_level':          'IPX Level',
    'charging_time_hrs':  'Charging Time (hrs)',
    'latency_ms':         'Latency (ms)',
    'range_m':            'Wireless Range (m)',
    'eq_modes':           'EQ Modes',
    # Binary (8 shown here — rest in next chart)
    'has_noise_cancellation': 'Has ANC',
    'has_enc':            'Has ENC',
    'has_fast_charge':    'Has Fast Charge',
    'has_gaming_mode':    'Has Gaming Mode',
    'has_touch_control':  'Has Touch Control',
    'has_low_latency':    'Has Low Latency',
    'has_dual_pairing':   'Has Dual Pairing',
    'has_spatial_audio':  'Has Spatial Audio',
}

rows = []
for col, label in features_18.items():
    if col in df.columns:
        non_null = int(df[col].notna().sum())
        non_zero = int((df[col].fillna(0) != 0).sum())
        pct = non_null / len(df) * 100
        rows.append({'Feature': label, 'Column': col, 'Non-Null': non_null, 'Non-Zero': non_zero, 'Coverage %': round(pct,1)})

cov_df = pd.DataFrame(rows).sort_values('Non-Zero', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#2ecc71' if v >= 50 else '#378ADD' if v >= 10 else '#e74c3c' for v in cov_df['Non-Zero']]
ax.barh(cov_df['Feature'], cov_df['Non-Zero'], color=colors)
ax.set_title('18 Scraped Features — Product Count with Feature Present', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Products (out of 219)')
ax.axvline(219*0.5, color='green', ls='--', alpha=0.5, label='50% threshold')
ax.legend()
for i, (v, pct) in enumerate(zip(cov_df['Non-Zero'], cov_df['Coverage %'])):
    ax.text(v+1, i, f'{v}  ({pct}%)', va='center', fontsize=9)
plt.tight_layout(); plt.show()

print("\nFull 18-feature coverage table:")
print(cov_df[['Feature','Non-Null','Non-Zero','Coverage %']].to_string(index=False))

## 6. Binary Features — Prevalence & Price Impact

In [ ]:
bin_features = {
    'has_noise_cancellation': 'ANC',
    'has_enc':                'ENC',
    'has_fast_charge':        'Fast Charge',
    'has_gaming_mode':        'Gaming Mode',
    'has_touch_control':      'Touch Control',
    'has_low_latency':        'Low Latency',
    'has_dual_pairing':       'Dual Pairing',
    'has_spatial_audio':      'Spatial Audio',
    'has_voice_assistant':    'Voice Assistant',
    'has_hi_res_audio':       'Hi-Res Audio',
    'has_usb_c':              'USB-C',
    'has_premium_codec':      'Premium Codec',
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Prevalence
counts = {label: int(df[col].sum()) for col, label in bin_features.items() if col in df.columns}
s = pd.Series(counts).sort_values(ascending=True)
axes[0].barh(s.index, s.values, color='#378ADD')
axes[0].set_title('Binary Feature Prevalence', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Number of Products')
for i,v in enumerate(s.values): axes[0].text(v+0.3, i, str(v), va='center', fontweight='bold')

# Price impact: median price with vs without each feature
price_impact = {}
for col, label in bin_features.items():
    if col in df.columns and df[col].sum() >= 3:
        yes = df[df[col]==1]['price_inr'].median()
        no  = df[df[col]==0]['price_inr'].median()
        price_impact[label] = yes - no

pi = pd.Series(price_impact).sort_values()
colors2 = ['#2ecc71' if v > 0 else '#e74c3c' for v in pi.values]
axes[1].barh(pi.index, pi.values, color=colors2)
axes[1].axvline(0, color='black', lw=1)
axes[1].set_title('Price Premium: With Feature vs Without (₹)', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Median Price Difference (INR)')
for i,v in enumerate(pi.values): axes[1].text(v + (20 if v>=0 else -20), i, f'₹{v:+.0f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout(); plt.show()

## 7. Numerical Features — Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

plots = [
    ('battery_life_hrs',  'Battery Life (hrs)',     '#1D9E75'),
    ('bluetooth_version', 'Bluetooth Version',      '#378ADD'),
    ('driver_size_mm',    'Driver Size (mm)',        '#D85A30'),
    ('ipx_level',         'IPX Water Resistance',   '#BA7517'),
    ('latency_ms',        'Latency (ms)',            '#D4537E'),
    ('anc_db',            'ANC Level (dB)',          '#888888'),
]

for ax, (col, label, color) in zip(axes, plots):
    data = df[col].dropna()
    if len(data) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label, fontweight='bold')
        continue
    ax.hist(data, bins=20, color=color, edgecolor='white')
    ax.axvline(data.median(), color='red', ls='--', lw=2, label=f'Median: {data.median():.1f}')
    ax.set_title(f'{label}  (n={len(data)})', fontweight='bold')
    ax.set_xlabel(label); ax.legend(fontsize=8)

plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("Numerical feature stats:")
for col in ['battery_life_hrs','bluetooth_version','driver_size_mm','ipx_level','latency_ms','anc_db','mic_count','charging_time_hrs','range_m','eq_modes']:
    if col in df.columns:
        s = df[col].dropna()
        if len(s) > 0:
            print(f"  {col:25s}: n={len(s):3d} | mean={s.mean():.1f} | median={s.median():.1f} | min={s.min():.1f} | max={s.max():.1f}")
        else:
            print(f"  {col:25s}: no data")

## 8. Battery Life & Price Relationship

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(df['battery_life_hrs'], df['price_inr'], alpha=0.5, color='#378ADD', s=60)
axes[0].set_title('Battery Life vs Price', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Battery Life (hours)'); axes[0].set_ylabel('Price (INR)')

# Battery buckets
df['battery_bucket'] = pd.cut(df['battery_life_hrs'], bins=[0,20,40,60,100,400],
                               labels=['<20h','20-40h','40-60h','60-100h','>100h'])
bp = df.groupby('battery_bucket', observed=True)['price_inr'].median()
axes[1].bar(bp.index.astype(str), bp.values, color='#1D9E75')
axes[1].set_title('Median Price by Battery Life Bucket', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Battery Life Range'); axes[1].set_ylabel('Median Price (INR)')
for i,v in enumerate(bp.values): axes[1].text(i, v+30, f'₹{v:.0f}', ha='center', fontweight='bold')

plt.tight_layout(); plt.show()
batt = df['battery_life_hrs'].dropna()
print(f"Battery: mean={batt.mean():.1f}h | median={batt.median():.1f}h | max={batt.max():.0f}h | n={len(batt)}")

## 9. Rating Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

rat = df['rating'].dropna()
axes[0].hist(rat, bins=20, color='#BA7517', edgecolor='white')
axes[0].axvline(rat.mean(), color='red', ls='--', lw=2, label=f'Mean: {rat.mean():.2f}')
axes[0].set_title(f'Rating Distribution  (n={len(rat)})', fontweight='bold'); axes[0].set_xlabel('Rating'); axes[0].legend()

axes[1].scatter(df['rating'], df['price_inr'], alpha=0.5, color='#BA7517', s=50)
axes[1].set_title('Rating vs Price', fontweight='bold'); axes[1].set_xlabel('Rating'); axes[1].set_ylabel('Price (INR)')

# Rating by category
df.boxplot(column='rating', by='category', ax=axes[2],
           boxprops=dict(color='#BA7517'), medianprops=dict(color='red', linewidth=2))
axes[2].set_title('Rating by Category', fontweight='bold'); axes[2].set_xlabel(''); plt.sca(axes[2]); plt.xticks(rotation=15); plt.suptitle('')

plt.tight_layout(); plt.show()
print(f"Rating: mean={rat.mean():.2f} | median={rat.median():.2f} | min={rat.min():.1f} | max={rat.max():.1f}")
print(f"Rated ≥ 4.0: {(rat>=4.0).sum()} products ({(rat>=4.0).mean()*100:.1f}%)")

## 10. Correlation Heatmap — All 18 Scraped Features

# Only keep columns with ≥10 non-null values
corr_cols = ['price_inr','rating','review_count','battery_life_hrs','bluetooth_version',
             'driver_size_mm','ipx_level','latency_ms','anc_db',
             'has_noise_cancellation','has_enc','has_fast_charge','has_gaming_mode',
             'has_touch_control','has_low_latency','has_dual_pairing',
             'has_spatial_audio','has_voice_assistant','has_hi_res_audio',
             'has_usb_c','has_premium_codec']
corr_cols = [c for c in corr_cols if c in df.columns and df[c].notna().sum() >= 10]

# Fill remaining NaNs with column median so every pair has full overlap → zero blank cells
corr_data = df[corr_cols].apply(pd.to_numeric, errors='coerce')
corr_data = corr_data.fillna(corr_data.median())
corr = corr_data.corr()

# Friendly short labels
labels = {
    'price_inr': 'Price', 'rating': 'Rating', 'review_count': 'Reviews',
    'battery_life_hrs': 'Battery (h)', 'bluetooth_version': 'BT Version',
    'driver_size_mm': 'Driver (mm)', 'ipx_level': 'IPX Level',
    'latency_ms': 'Latency (ms)', 'anc_db': 'ANC (dB)',
    'has_noise_cancellation': 'ANC', 'has_enc': 'ENC',
    'has_fast_charge': 'Fast Charge', 'has_gaming_mode': 'Gaming Mode',
    'has_touch_control': 'Touch Ctrl', 'has_low_latency': 'Low Latency',
    'has_dual_pairing': 'Dual Pair', 'has_spatial_audio': 'Spatial Audio',
    'has_voice_assistant': 'Voice Asst', 'has_hi_res_audio': 'Hi-Res',
    'has_usb_c': 'USB-C', 'has_premium_codec': 'Codec',
}
corr.index   = [labels.get(c, c) for c in corr.index]
corr.columns = [labels.get(c, c) for c in corr.columns]

plt.figure(figsize=(16, 13))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.75, 'label': 'Pearson r'},
            annot_kws={'size': 8})
plt.title('Correlation Heatmap — 18 Scraped Features + Price', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout(); plt.show()

print(f"Heatmap: {corr.shape[0]} × {corr.shape[1]} — zero blank cells")
print("\nCorrelation with Price (sorted):")
price_corr = corr['Price'].drop('Price').sort_values(ascending=False)
print(price_corr.round(3).to_string())

## 11. Key EDA Insights Summary

In [ ]:
print("=" * 65)
print("  KEY EDA INSIGHTS — NexTune Bluetooth Headphones")
print("=" * 65)
print(f"\n📦 Dataset: {len(df)} products | {df['brand'].nunique()} brands | 40 features")
print(f"\n💰 Price")
print(f"   ₹{df['price_inr'].min():.0f} – ₹{df['price_inr'].max():.0f}  |  Median ₹{df['price_inr'].median():.0f}  |  Mean ₹{df['price_inr'].mean():.0f}")
print(f"   75% of products priced below ₹{df['price_inr'].quantile(0.75):.0f}")
print(f"\n🏷️  Categories")
for cat,cnt in df['category'].value_counts().items():
    print(f"   {cat}: {cnt} ({cnt/len(df)*100:.1f}%)")
print(f"\n🌍 Country of Origin")
for c,cnt in df['country_of_origin'].value_counts().items():
    print(f"   {c}: {cnt}")
print(f"\n�� Battery Life  (n={df['battery_life_hrs'].notna().sum()})")
batt=df['battery_life_hrs'].dropna()
print(f"   Mean {batt.mean():.1f}h | Median {batt.median():.1f}h | Max {batt.max():.0f}h")
print(f"\n⭐ Ratings  (n={df['rating'].notna().sum()})")
rat=df['rating'].dropna()
print(f"   Mean {rat.mean():.2f} | {(rat>=4.0).sum()} products rated ≥ 4.0")
print(f"\n🎯 18 Scraped Features — Key Stats")
anc_med = df[df['has_noise_cancellation']==1]['price_inr'].median()
no_anc  = df[df['has_noise_cancellation']==0]['price_inr'].median()
print(f"   ANC: {int(df['has_noise_cancellation'].sum())} products  →  median ₹{anc_med:.0f} vs ₹{no_anc:.0f} without  (+₹{anc_med-no_anc:.0f} premium)")
print(f"   ENC: {int(df['has_enc'].sum())} | Fast Charge: {int(df['has_fast_charge'].sum())} | Gaming Mode: {int(df['has_gaming_mode'].sum())}")
print(f"   Touch Control: {int(df['has_touch_control'].sum())} | Low Latency: {int(df['has_low_latency'].sum())} | Spatial Audio: {int(df['has_spatial_audio'].sum())}")
print(f"   Bluetooth avg: v{df['bluetooth_version'].mean():.1f} | Driver avg: {df['driver_size_mm'].mean():.1f}mm | IPX avg: {df['ipx_level'].mean():.1f}")
print("=" * 65)